# NB04a — Semillas adicionales para A0 y A1

**MIA-07 · Examen Final · Proyecto Crack-Seg · Etapa 4 (Integrante 4)**

**¿Por qué este notebook?** En VAL, A1 supera a A0 por solo **+0.006 de Dice**, y el historial de NB03
muestra que el Dice de validación oscila **±0.015 de una epoch a otra**. Con **una sola semilla** no
podemos saber si esa diferencia es real o es suerte de la inicialización. El requisito **R4** del examen
pide *"varias semillas (media ± desviación)"*.

**Qué hace:** entrena **un modelo (A0 o A1) con una semilla**, con el **mismo protocolo** de NB02/NB03.
Cada integrante ejecuta una combinación; las 4 corridas pueden ir **en paralelo** en cuentas distintas de Colab.

| Corrida | MODELO | SEED | Responsable sugerido | Tiempo aprox. (T4) |
|---|---|---|---|---|
| ya existe (NB02) | A0 | 20261026 | Integrante 2 | 52 min |
| ya existe (NB03) | A1 | 20261026 | Integrante 3 | ~28 min en L4 |
| **nueva** | A0 | 20261027 | Integrante 1 | ~50–60 min |
| **nueva** | A0 | 20261028 | Integrante 2 | ~50–60 min |
| **nueva** | A1 | 20261027 | Integrante 3 | ~50–60 min |
| **nueva** | A1 | 20261028 | Integrante 4 | ~50–60 min |

**Instrucciones:**
1. Entorno de ejecución → Cambiar tipo → **GPU T4** (todas las corridas nuevas en T4).
2. En la **Celda 2** cambiar solo `MODELO` y `SEED`.
3. Ejecutar las Celdas 1 → 6. Si Colab se desconecta, volver a ejecutar 1 → 5: se **reanuda** desde `checkpoint_last.pt`.

> **Qué varía la semilla:** los pesos iniciales de la red. El orden de TRAIN lo fija el lector de Nicolás
> (seed interna 42) y es igual en todas las corridas. Por tanto medimos la variabilidad **debida a la inicialización**;
> esto se declara en el artículo.
>
> TEST **no** se usa aquí.

In [ ]:
# ============================================================
# NB04a - CELDA 1: VERIFICACIÓN DEL ENTORNO
# ============================================================

import sys, platform
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

if torch.cuda.is_available():
    NOMBRE_GPU = torch.cuda.get_device_name(0)
    print("GPU:", NOMBRE_GPU)
    if "T4" not in NOMBRE_GPU:
        print("AVISO: el protocolo pide T4 para que los tiempos sean comparables.")
        print("       Puedes continuar (las MÉTRICAS no dependen de la GPU),")
        print("       pero el tiempo quedará registrado con esta GPU.")
else:
    NOMBRE_GPU = "CPU"
    print("AVISO: sin GPU. Solo sirve para PRUEBA_LOCAL = True.")

print("\nEstado: PASS - Entorno identificado.")

In [ ]:
# ============================================================
# NB04a - CELDA 2: CONFIGURACIÓN (EDITAR SOLO MODELO Y SEED)
# ============================================================

import sys
from pathlib import Path

# ---------------- EDITAR AQUÍ ----------------
MODELO = "A1"          # "A0" (sin skip) o "A1" (Mini U-Net)
SEED = 20261027        # 20261027 o 20261028 (20261026 ya existe)
# ---------------------------------------------

# Solo para probar el flujo en CPU con pocos datos.
# En Colab SIEMPRE False.
PRUEBA_LOCAL = False

assert MODELO in ("A0", "A1")
assert SEED != 20261026, "La semilla 20261026 ya está entrenada en NB02/NB03."

# Protocolo IDÉNTICO a NB02/NB03.
BATCH_SIZE = 4
IMAGE_SIZE = 416
LEARNING_RATE = 0.001
MAX_EPOCHS = 50
PATIENCE = 7
MIN_DELTA = 0.001
UMBRAL = 0.5
NOMBRE_DATASET = "Crack-Seg completo v1"

if PRUEBA_LOCAL:
    MAX_EPOCHS = 2
    MAX_LOTES = 3      # minibatches por epoch en la prueba
else:
    MAX_LOTES = None

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/MIA-Grupo3/MIA-07_DL/ExFinal_Fisuras")
    DATOS_LOCALES = Path("/content/datos_NB04")
    TEMPORAL_LOCAL = Path(f"/content/tmp_{MODELO}_{SEED}.pt")
else:
    DRIVE_BASE = Path.cwd().resolve().parent
    DATOS_LOCALES = DRIVE_BASE / "data" / "completo"
    TEMPORAL_LOCAL = DRIVE_BASE / f"tmp_{MODELO}_{SEED}.pt"

ZIP_DATOS = DRIVE_BASE / "Entrega_Victor_Datos_Completos_v1.zip"
RUTA_DATOS = DATOS_LOCALES / "Entrega_Victor_Datos_Completos_v1"

# Carpeta propia de esta corrida: nunca toca NB02 ni NB03.
CARPETA_RESULTADOS = DRIVE_BASE / "resultados" / "NB04_Semillas" / f"{MODELO}_seed{SEED}"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_LAST = CARPETA_RESULTADOS / "checkpoint_last.pt"
CHECKPOINT_BEST = CARPETA_RESULTADOS / "checkpoint_best.pt"

assert "NB02" not in str(CARPETA_RESULTADOS) and "NB03" not in str(CARPETA_RESULTADOS)

print("Corrida:", MODELO, "| seed", SEED)
print("Resultados:", CARPETA_RESULTADOS)
print(f"Batch {BATCH_SIZE} | {IMAGE_SIZE}px | lr {LEARNING_RATE} | ES {MAX_EPOCHS}/{PATIENCE}/{MIN_DELTA}")
print("\nEstado: PASS - Configuración lista.")

In [ ]:
# ============================================================
# NB04a - CELDA 3: ARQUITECTURAS A0 Y A1 + PÉRDIDA (copiadas de NB03)
#
# Se copian SIN cambios desde NB03 (Celdas 3 y 4). Si se
# modificara una sola línea, los checkpoints dejarían de cargar
# con strict=True, y eso nos avisaría del error.
# ============================================================

import torch
import torch.nn as nn


class BloqueConv(nn.Module):
    """Dos convoluciones 3x3 + ReLU (idéntico a NB02/NB03)."""

    def __init__(self, canales_entrada, canales_salida):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Conv2d(canales_entrada, canales_salida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(canales_salida, canales_salida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.capas(x)


class EncoderDecoder(nn.Module):
    """A0: Encoder-Decoder SIN conexiones skip (NB02)."""

    def __init__(self):
        super().__init__()
        self.encoder1 = BloqueConv(3, 16)
        self.pool1 = nn.MaxPool2d(2)
        self.encoder2 = BloqueConv(16, 32)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = BloqueConv(32, 64)
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.decoder1 = BloqueConv(64, 32)
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.decoder2 = BloqueConv(32, 16)
        self.salida = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        x = self.encoder1(x)
        x = self.pool1(x)
        x = self.encoder2(x)
        x = self.pool2(x)
        x = self.bottleneck(x)
        x = self.up1(x)
        x = self.decoder1(x)
        x = self.up2(x)
        x = self.decoder2(x)
        return self.salida(x)


class MiniUNet(nn.Module):
    """A1: misma red que A0 + 2 conexiones skip por concatenación (NB03)."""

    def __init__(self):
        super().__init__()
        self.encoder1 = BloqueConv(3, 16)
        self.pool1 = nn.MaxPool2d(2)
        self.encoder2 = BloqueConv(16, 32)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = BloqueConv(32, 64)
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.decoder1 = BloqueConv(64 + 32, 32)
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.decoder2 = BloqueConv(32 + 16, 16)
        self.salida = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        skip1 = self.encoder1(x)
        x = self.pool1(skip1)
        skip2 = self.encoder2(x)
        x = self.pool2(skip2)
        x = self.bottleneck(x)
        x = self.up1(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.decoder1(x)
        x = self.up2(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.decoder2(x)
        return self.salida(x)


class BCEDiceLoss(nn.Module):
    """BCE + Dice (1:1), idéntica a NB02/NB03."""

    def __init__(self, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, mascaras):
        loss_bce = self.bce(logits, mascaras)
        prob = torch.sigmoid(logits)
        dims = (1, 2, 3)
        inter = torch.sum(prob * mascaras, dim=dims)
        dice = (2 * inter + self.smooth) / (
            torch.sum(prob, dim=dims) + torch.sum(mascaras, dim=dims) + self.smooth
        )
        return loss_bce + (1 - dice.mean())


def contar_parametros(red):
    return sum(p.numel() for p in red.parameters() if p.requires_grad)


# Diccionario que conecta el nombre corto con la clase.
CLASES = {"A0": EncoderDecoder, "A1": MiniUNet}
NOMBRES = {"A0": "A0 Encoder-Decoder (sin skip)", "A1": "A1 Mini U-Net (con skip)"}
ARQUITECTURA = {"A0": "EncoderDecoder_sin_skip", "A1": "MiniUNet_con_skip"}

assert contar_parametros(EncoderDecoder()) == 106753
assert contar_parametros(MiniUNet()) == 118273
print("A0:", f"{contar_parametros(EncoderDecoder()):,}", "parámetros")
print("A1:", f"{contar_parametros(MiniUNet()):,}", "parámetros")
print("\nEstado: PASS - Arquitecturas idénticas a NB02/NB03.")

In [ ]:
# ============================================================
# NB04a - CELDA 4: EXTRACCIÓN DEL ZIP Y LECTOR DE NICOLÁS (igual que NB03)
# ============================================================

import sys
import zipfile
import importlib.util

if not (RUTA_DATOS / "lector_pytorch.py").is_file():
    assert ZIP_DATOS.is_file(), f"No se encontró el ZIP: {ZIP_DATOS}"
    print("Extrayendo", ZIP_DATOS.name, "...")
    DATOS_LOCALES.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_DATOS, "r") as z:
        z.extractall(DATOS_LOCALES)
    print("Extracción finalizada.")

assert (RUTA_DATOS / "lector_pytorch.py").is_file(), (
    f"No se encontró lector_pytorch.py en {RUTA_DATOS}"
)

if str(RUTA_DATOS) not in sys.path:
    sys.path.insert(0, str(RUTA_DATOS))

spec = importlib.util.spec_from_file_location(
    "lector_pytorch_nb04", str(RUTA_DATOS / "lector_pytorch.py")
)
modulo_lector = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = modulo_lector
spec.loader.exec_module(modulo_lector)

# Mismos argumentos que NB02/NB03 (el lector baraja TRAIN con
# seed interna 42 y NO baraja VAL).
train_loader, val_loader = modulo_lector.make_loaders(
    RUTA_DATOS, batch_size=BATCH_SIZE, size=IMAGE_SIZE
)

if not PRUEBA_LOCAL:
    assert len(train_loader.dataset) == 3717
    assert len(val_loader.dataset) == 199

print("TRAIN:", len(train_loader.dataset), "| VAL:", len(val_loader.dataset))
print("\nEstado: PASS - Datos listos (mismas particiones que NB02/NB03).")

In [ ]:
# ============================================================
# NB04a - CELDA 5: ENTRENAMIENTO RECUPERABLE (lógica de NB03 Celda 6)
#
# Mismo bucle que NB03: Adam, BCE+Dice, Early Stopping sobre
# Val Loss, checkpoints BEST/LAST en Drive y reanudación.
# ============================================================

import os, time, random, shutil, itertools
from datetime import datetime
import numpy as np
import torch

if not PRUEBA_LOCAL and not torch.cuda.is_available():
    raise RuntimeError("Se requiere GPU. Activar T4 en Colab.")

usa_gpu = device.type == "cuda"

def sincronizar():
    if usa_gpu:
        torch.cuda.synchronize()

# 1. Semillas ANTES de crear el modelo (mismo orden que NB03).
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if usa_gpu:
    torch.cuda.manual_seed_all(SEED)

modelo = CLASES[MODELO]().to(device)
criterio = BCEDiceLoss().to(device)
optimizador = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
cantidad_parametros = contar_parametros(modelo)

historial = []
mejor_val_loss = float("inf")
epochs_sin_mejora = 0
ultima_epoch = 0
tiempo_acumulado = 0.0


def guardar_checkpoint(ruta, epoch_actual):
    estado = {
        "epoch": epoch_actual,
        "arquitectura": ARQUITECTURA[MODELO],
        "modelo_corto": MODELO,
        "model_state_dict": modelo.state_dict(),
        "optimizer_state_dict": optimizador.state_dict(),
        "best_val_loss": mejor_val_loss,
        "epochs_sin_mejora": epochs_sin_mejora,
        "historial": historial,
        "tiempo_acumulado_seg": tiempo_acumulado,
        "seed": SEED,
        "parametros": cantidad_parametros,
        "dataset": NOMBRE_DATASET,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "gpu": NOMBRE_GPU,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if usa_gpu else None,
    }
    # Guardado seguro: primero local, luego copia y reemplazo atómico.
    torch.save(estado, TEMPORAL_LOCAL)
    tmp = ruta.with_name(ruta.name + ".tmp")
    shutil.copyfile(TEMPORAL_LOCAL, tmp)
    os.replace(tmp, ruta)


# 2. Reanudar si existe checkpoint LAST.
if CHECKPOINT_LAST.is_file():
    estado = torch.load(CHECKPOINT_LAST, map_location=device, weights_only=False)
    assert estado["arquitectura"] == ARQUITECTURA[MODELO]
    assert estado["seed"] == SEED
    modelo.load_state_dict(estado["model_state_dict"])
    optimizador.load_state_dict(estado["optimizer_state_dict"])
    ultima_epoch = estado["epoch"]
    mejor_val_loss = estado["best_val_loss"]
    epochs_sin_mejora = estado["epochs_sin_mejora"]
    historial = estado["historial"]
    tiempo_acumulado = estado["tiempo_acumulado_seg"]
    random.setstate(estado["python_rng"])
    np.random.set_state(estado["numpy_rng"])
    torch.set_rng_state(estado["torch_rng"].cpu())
    if usa_gpu and estado["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all(estado["cuda_rng"])
    print(f"Reanudando desde la epoch {ultima_epoch + 1} "
          f"(mejor Val Loss {mejor_val_loss:.6f}).")
else:
    print("Sin checkpoints previos: se empieza en la epoch 1.")

print("=" * 70)
print(f"ENTRENAMIENTO {NOMBRES[MODELO]} | seed {SEED} | {NOMBRE_GPU}")
print("Inicio:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("=" * 70)

inicio_epoch = ultima_epoch + 1
if epochs_sin_mejora >= PATIENCE:
    print("Early Stopping ya alcanzado en una sesión anterior.")
    inicio_epoch = MAX_EPOCHS + 1

for epoch in range(inicio_epoch, MAX_EPOCHS + 1):

    if usa_gpu:
        torch.cuda.reset_peak_memory_stats()
    sincronizar()
    t0 = time.perf_counter()

    # --- Entrenamiento ---
    modelo.train()
    suma, n_tot = 0.0, 0
    for imagenes, mascaras, _ in itertools.islice(train_loader, MAX_LOTES):
        imagenes, mascaras = imagenes.to(device), mascaras.to(device)
        optimizador.zero_grad(set_to_none=True)
        loss = criterio(modelo(imagenes), mascaras)
        loss.backward()
        optimizador.step()
        suma += loss.item() * imagenes.size(0)
        n_tot += imagenes.size(0)
    loss_train = suma / n_tot
    sincronizar()
    t_train = time.perf_counter() - t0

    # --- Validación (Dice/IoU globales, misma fórmula que NB02) ---
    modelo.eval()
    suma, n_tot, tp, fp, fn = 0.0, 0, 0, 0, 0
    t1 = time.perf_counter()
    with torch.no_grad():
        for imagenes, mascaras, _ in itertools.islice(val_loader, MAX_LOTES):
            imagenes, mascaras = imagenes.to(device), mascaras.to(device)
            logits = modelo(imagenes)
            suma += criterio(logits, mascaras).item() * imagenes.size(0)
            n_tot += imagenes.size(0)
            pred = torch.sigmoid(logits) >= UMBRAL
            real = mascaras >= 0.5
            tp += int((pred & real).sum())
            fp += int((pred & ~real).sum())
            fn += int((~pred & real).sum())
    loss_val = suma / n_tot
    sincronizar()
    t_val = time.perf_counter() - t1

    tiempo_epoch = t_train + t_val
    tiempo_acumulado += tiempo_epoch
    memoria = torch.cuda.max_memory_allocated() / 1024**2 if usa_gpu else 0.0

    mejora = loss_val < (mejor_val_loss - MIN_DELTA)
    if mejora:
        mejor_val_loss = loss_val
        epochs_sin_mejora = 0
    else:
        epochs_sin_mejora += 1

    historial.append({
        "epoch": epoch,
        "train_loss": loss_train,
        "val_loss": loss_val,
        "val_dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "val_iou": tp / max(tp + fp + fn, 1),
        "val_precision": tp / max(tp + fp, 1),
        "val_recall": tp / max(tp + fn, 1),
        "tiempo_train_seg": t_train,
        "tiempo_val_seg": t_val,
        "tiempo_epoch_seg": tiempo_epoch,
        "memoria_gpu_pico_mb": memoria,
        "mejoro": mejora,
    })

    if mejora:
        guardar_checkpoint(CHECKPOINT_BEST, epoch)
    guardar_checkpoint(CHECKPOINT_LAST, epoch)

    r = historial[-1]
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | Train {loss_train:.4f} | Val {loss_val:.4f} | "
          f"Dice {r['val_dice']:.4f} | IoU {r['val_iou']:.4f} | {tiempo_epoch:.1f}s | "
          f"{memoria:.0f} MB | Paciencia {epochs_sin_mejora}/{PATIENCE}"
          + (" | BEST" if mejora else ""), flush=True)

    if epochs_sin_mejora >= PATIENCE:
        print("\nEARLY STOPPING.")
        break

print("\nTiempo acumulado:", round(tiempo_acumulado / 60, 2), "min")
print("Estado: PASS - Entrenamiento terminado.")

In [ ]:
# ============================================================
# NB04a - CELDA 6: EXPORTAR HISTORIAL Y RESUMEN DE LA CORRIDA
# (Métricas de VALIDACIÓN; TEST se evalúa solo en NB04b.)
# ============================================================

import json
import pandas as pd

estado_last = torch.load(CHECKPOINT_LAST, map_location="cpu", weights_only=False)
estado_best = torch.load(CHECKPOINT_BEST, map_location="cpu", weights_only=False)
df_hist = pd.DataFrame(estado_last["historial"])
fila_best = df_hist[df_hist["epoch"] == estado_best["epoch"]].iloc[0]

df_hist.to_csv(CARPETA_RESULTADOS / "Historial_Entrenamiento.csv", index=False, encoding="utf-8-sig")

resumen = {
    "modelo": MODELO,
    "arquitectura": ARQUITECTURA[MODELO],
    "seed": SEED,
    "parametros": cantidad_parametros,
    "gpu": estado_last["gpu"],
    "epochs_ejecutadas": int(len(df_hist)),
    "epoch_best": int(estado_best["epoch"]),
    "val_loss_best": float(fila_best["val_loss"]),
    "val_dice_best": float(fila_best["val_dice"]),
    "val_iou_best": float(fila_best["val_iou"]),
    "tiempo_total_min": float(estado_last["tiempo_acumulado_seg"] / 60),
    "tiempo_medio_epoch_seg": float(df_hist["tiempo_epoch_seg"].mean()),
    "memoria_gpu_pico_mb": float(df_hist["memoria_gpu_pico_mb"].max()),
    "metricas": "VALIDACIÓN (no TEST)",
}
with open(CARPETA_RESULTADOS / "Resumen_Corrida.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=4, ensure_ascii=False)

for k, v in resumen.items():
    print(f"{k:>24}: {v}")
print("\nEstado: PASS - Corrida exportada. Avisar al Integrante 4.")